# FOL Reasoning Benchmarks Demo

This notebook demonstrates a neuro-symbolic pipeline for converting natural language reasoning tasks into first-order logic (FOL) representations. We evaluate on three standardized benchmarks:

1. **RuleTaker** — Logical entailment with explicit rules and facts
2. **FOLIO** — First-order logic implications with premises and conclusions  
3. **ProofWriter** — Theories with rules and queries requiring multi-hop reasoning

The data is standardized to `exp_sel_data_out` schema, enabling consistent evaluation across datasets.

## Setup: Install dependencies

Install packages needed for data processing and visualization. On Colab, core packages (numpy, pandas) are pre-installed; locally we install at Colab's exact versions.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Non-Colab packages (always install)
_pip('loguru==0.7.2')

# Core packages — pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'tabulate==0.9.0')

print("✓ All dependencies installed")

## Imports

Load libraries for data processing, logging, and visualization.

In [ ]:
import json
import sys
from pathlib import Path
from loguru import logger
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from tabulate import tabulate

# Configure logging
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

print("✓ All imports loaded")

## Data Loading

Load the demo dataset from GitHub with local fallback. This pattern works both locally and in Colab.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-14efe0-resolution-failure-directed-extraction-d/main/round-1/dataset-1/demo/mini_demo_data.json"
import os

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local path")

logger.info("Data loading helper defined")

## Load Dataset

Load the mini demo data (9 examples total: 3 RuleTaker, 3 FOLIO, 3 ProofWriter).

In [ ]:
data = load_data()
logger.info(f"Loaded {len(data['datasets'])} datasets")
for ds in data['datasets']:
    logger.info(f"  {ds['dataset']}: {len(ds['examples'])} examples")

## Configuration

Define tunable parameters for processing and analysis. Start with minimum values for quick demo; scale up as needed.

In [ ]:
# ========== CONFIGURATION ==========
# For full run, increase these values:
# MAX_EXAMPLES_PER_DATASET = 5000
# TEXT_LENGTH_LIMIT = 5000

MAX_EXAMPLES_PER_DATASET = 3      # Process first N examples per dataset
TEXT_LENGTH_LIMIT = 500           # Truncate inputs/outputs for display
EXAMPLE_DISPLAY_LIMIT = 2         # Show first N examples per dataset

logger.info(f"Config: max_examples={MAX_EXAMPLES_PER_DATASET}, text_limit={TEXT_LENGTH_LIMIT}")

## Standardize and Validate Data

Verify all examples conform to the `exp_sel_data_out` schema: `input`, `output`, and `metadata_*` fields.

In [ ]:
def standardize_datasets(data: dict, max_per_dataset: int = None) -> list:
    """Standardize datasets to exp_sel_data_out schema."""
    all_examples = []
    for ds_dict in data['datasets']:
        dataset_name = ds_dict['dataset']
        examples = ds_dict['examples']
        
        if max_per_dataset:
            examples = examples[:max_per_dataset]
        
        for ex in examples:
            # Validate required fields
            if 'input' not in ex or 'output' not in ex:
                logger.warning(f"Skipping invalid example in {dataset_name}")
                continue
            
            # Build standardized example
            standardized = {
                'dataset': dataset_name,
                'input': ex['input'],
                'output': ex['output'],
            }
            
            # Add all metadata fields
            for key, val in ex.items():
                if key.startswith('metadata_'):
                    standardized[key] = val
            
            all_examples.append(standardized)
    
    return all_examples

examples = standardize_datasets(data, max_per_dataset=MAX_EXAMPLES_PER_DATASET)
logger.info(f"Standardized {len(examples)} examples")

## Dataset Analysis

Analyze structure, task types, and output label distributions.

In [ ]:
# Count examples per dataset
dataset_counts = Counter(ex['dataset'] for ex in examples)
logger.info(f"Examples per dataset: {dict(dataset_counts)}")

# Count task types
task_types = Counter(ex.get('metadata_task_type', 'unknown') for ex in examples)
logger.info(f"Task types: {dict(task_types)}")

# Count output labels
output_labels = Counter(ex['output'] for ex in examples)
logger.info(f"Output labels: {dict(output_labels)}")

# Compute text statistics
input_lengths = [len(ex['input']) for ex in examples]
logger.info(f"Input length: mean={sum(input_lengths)//len(input_lengths)}, min={min(input_lengths)}, max={max(input_lengths)}")

## Display Example Examples

Show a few representative examples from each dataset.

In [ ]:
for dataset_name in dataset_counts.keys():
    dataset_examples = [ex for ex in examples if ex['dataset'] == dataset_name][:EXAMPLE_DISPLAY_LIMIT]
    print(f"\n{'='*70}")
    print(f"Dataset: {dataset_name}")
    print(f"{'='*70}")
    
    for i, ex in enumerate(dataset_examples, 1):
        print(f"\nExample {i}:")
        print(f"Input ({len(ex['input'])} chars):")
        print(f"  {ex['input'][:TEXT_LENGTH_LIMIT]}..." if len(ex['input']) > TEXT_LENGTH_LIMIT else f"  {ex['input']}")
        print(f"Output: {ex['output']}")
        print(f"Task: {ex.get('metadata_task_type', 'N/A')}")

## FOL Annotation Analysis

Examine first-order logic annotations where available (FOLIO dataset).

In [ ]:
folio_examples = [ex for ex in examples if ex['dataset'] == 'FOLIO']
if folio_examples:
    print(f"\n{'='*70}")
    print("FOLIO: First-Order Logic Annotations")
    print(f"{'='*70}")
    
    for i, ex in enumerate(folio_examples[:EXAMPLE_DISPLAY_LIMIT], 1):
        print(f"\nExample {i}:")
        print(f"Premises (FOL):")
        fol = ex.get('metadata_premises_fol', 'N/A')
        print(f"  {fol[:200]}..." if len(str(fol)) > 200 else f"  {fol}")
        print(f"Conclusion (FOL):")
        conc = ex.get('metadata_conclusion_fol', 'N/A')
        print(f"  {conc}")
        print(f"Label: {ex['output']}")

## Proof and Reasoning Depth Analysis

Analyze multi-hop reasoning depth and proof structure across datasets.

In [ ]:
# Collect reasoning depth statistics
depths = []
for ex in examples:
    depth_str = ex.get('metadata_hop_depth', '0')
    try:
        depths.append(int(depth_str))
    except ValueError:
        pass

if depths:
    depth_counter = Counter(depths)
    print(f"\n{'='*70}")
    print("Reasoning Depth (Hops)")
    print(f"{'='*70}")
    for depth in sorted(depth_counter.keys()):
        print(f"Depth {depth}: {depth_counter[depth]} examples")
    print(f"Mean depth: {sum(depths)/len(depths):.2f}")
    print(f"Max depth: {max(depths)}")

# ProofWriter proof structure
proofwriter_examples = [ex for ex in examples if ex['dataset'] == 'ProofWriter']
if proofwriter_examples:
    print(f"\n{'='*70}")
    print("ProofWriter: Facts and Rules")
    print(f"{'='*70}")
    for i, ex in enumerate(proofwriter_examples[:EXAMPLE_DISPLAY_LIMIT], 1):
        print(f"\nExample {i}:")
        print(f"  Facts: {ex.get('metadata_n_facts', 'N/A')}")
        print(f"  Rules: {ex.get('metadata_n_rules', 'N/A')}")
        print(f"  Config: {ex.get('metadata_config', 'N/A')}")
        print(f"  Query result: {ex['output']}")

## Schema Validation

Verify all examples conform to `exp_sel_data_out` schema requirements.

In [ ]:
def validate_schema(examples: list) -> dict:
    """Validate examples against exp_sel_data_out schema."""
    required_fields = {'input', 'output'}
    all_fields = set()
    valid_count = 0
    invalid_examples = []
    
    for ex in examples:
        if required_fields.issubset(set(ex.keys())):
            valid_count += 1
        else:
            invalid_examples.append(ex)
        all_fields.update(ex.keys())
    
    return {
        'total': len(examples),
        'valid': valid_count,
        'invalid': len(invalid_examples),
        'all_fields': sorted(all_fields),
    }

validation = validate_schema(examples)
print(f"\n{'='*70}")
print("Schema Validation Report")
print(f"{'='*70}")
print(f"Total examples: {validation['total']}")
print(f"Valid (have 'input' + 'output'): {validation['valid']}")
print(f"Invalid: {validation['invalid']}")
print(f"\nAll fields present:")
for field in validation['all_fields']:
    print(f"  - {field}")

## Visualization and Summary

Summarize key metrics and create plots.

In [ ]:
# Create summary table
summary_data = []
for dataset_name in sorted(dataset_counts.keys()):
    ds_examples = [ex for ex in examples if ex['dataset'] == dataset_name]
    task_type = ds_examples[0].get('metadata_task_type', 'N/A') if ds_examples else 'N/A'
    avg_input_len = sum(len(ex['input']) for ex in ds_examples) // len(ds_examples) if ds_examples else 0
    outputs = Counter(ex['output'] for ex in ds_examples)
    summary_data.append([
        dataset_name,
        len(ds_examples),
        task_type,
        avg_input_len,
        str(dict(outputs))
    ])

headers = ['Dataset', 'Examples', 'Task Type', 'Avg Input Len', 'Output Labels']
print(f"\n{'='*70}")
print("Dataset Summary")
print(f"{'='*70}")
print(tabulate(summary_data, headers=headers, tablefmt='grid'))

# Plot dataset distribution
if len(dataset_counts) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Plot 1: Examples per dataset
    axes[0].bar(dataset_counts.keys(), dataset_counts.values(), color='steelblue')
    axes[0].set_title('Examples per Dataset')
    axes[0].set_ylabel('Count')
    axes[0].set_xlabel('Dataset')
    for i, (k, v) in enumerate(dataset_counts.items()):
        axes[0].text(i, v + 0.1, str(v), ha='center')
    
    # Plot 2: Task types distribution
    task_type_counts = Counter(ex.get('metadata_task_type', 'unknown') for ex in examples)
    axes[1].barh(list(task_type_counts.keys()), list(task_type_counts.values()), color='coral')
    axes[1].set_title('Task Type Distribution')
    axes[1].set_xlabel('Count')
    for i, (k, v) in enumerate(task_type_counts.items()):
        axes[1].text(v + 0.1, i, str(v), va='center')
    
    plt.tight_layout()
    plt.savefig('dataset_summary.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("✓ Visualization saved to dataset_summary.png")

print(f"\n{'='*70}")
print(f"Demo Summary: {len(examples)} total examples from {len(dataset_counts)} datasets")
print(f"All examples valid: {validation['valid'] == validation['total']}")
print(f"{'='*70}")